# Gauthier_Cerema : Test Steger & Frangi

Ce notebook implémente les premiers tests demandés :
1. Téléchargement des données via `gdown`.
2. Téléchargement ciblé du script `steger_gpu.py` depuis GitHub (pas de clonage complet).
3. Filtre Frangi classique (skimage) avec $\sigma=30$.
4. Calcul du Hessien Gaussien (implémentation GPU via PyTorch) avec $\sigma=30$ sur l'image spécifique 'Ombrage 05m base.tif'.

In [ ]:
!pip install rasterio torch torchvision gdown

# 1. Téléchargement du fichier nécessaire depuis GitHub
steger_url = "https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/steger_gpu.py"
!wget {steger_url} -O steger_gpu.py

graph_url = "https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/graph_builder.py"
!wget {graph_url} -O graph_builder.py

# 2. Téléchargement des données (gdown)
# ID du dossier Drive : 1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ
!gdown --folder https://drive.google.com/drive/folders/1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ -O Gauthier_Cerema_Data

In [ ]:
import sys
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage.filters import frangi
import torch

# Import depuis le fichier téléchargé localement
try:
    from steger_gpu import StegerHessian
    print("Module StegerHessian importé avec succès.")
except ImportError as e:
    print(f"Erreur d'import : {e}. Vérifiez que steger_gpu.py a bien été téléchargé.")

In [ ]:
# --- DEFINITION DU CHEMIN ---
# Les images sont dans le sous-dossier 'Premier_test' du téléchargement
base_data_path = 'Gauthier_Cerema_Data'
folder_path = os.path.join(base_data_path, 'Premier_test')

if os.path.exists(folder_path):
    print(f"Dossier trouvé : {folder_path}")
    files = os.listdir(folder_path)
    tif_files = [f for f in files if f.lower().endswith('.tif')]
    print(f"Fichiers .tif trouvés ({len(tif_files)}) :", tif_files)
    
    # --- Affichage des images brutes ---
    for f in tif_files:
        f_path = os.path.join(folder_path, f)
        try:
            with rasterio.open(f_path) as src:
                img = src.read(1)
                plt.figure(figsize=(6, 6))
                plt.imshow(img, cmap='gray')
                plt.title(f"Image brute : {f}")
                plt.axis('off')
                plt.show()
        except Exception as e:
            print(f"Impossible de lire {f} : {e}")
else:
    print(f"ATTENTION : Le dossier {folder_path} n'a pas été trouvé.")
    if os.path.exists(base_data_path):
        print(f"Contenu de {base_data_path} :", os.listdir(base_data_path))

In [ ]:
# --- 1. FILTRE FRANGI CLASSIQUE (tous les tifs) ---
σ_val = 30

if os.path.exists(folder_path):
    tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

    for file_name in tif_files:
        file_path = os.path.join(folder_path, file_name)
        try:
            with rasterio.open(file_path) as src:
                data = src.read(1)
                
                # Frangi Classique
                print(f"Traitement Frangi (σ={σ_val}) sur {file_name}...")
                response = frangi(data, sigmas=[σ_val])

                plt.figure(figsize=(10, 6))
                plt.imshow(response, cmap='gray')
                plt.colorbar(label='Frangi Response')
                plt.title(f"Frangi (σ={σ_val}) : {file_name}")
                plt.show()
        except Exception as e:
            print(f"Erreur lecture {file_name}: {e}")

In [ ]:
# --- 2. HESSIEN GAUSSIEN PONDÉRÉ (Fusion) ---
weights = {"Ombrage 05m base.tif": 0.5, "MNT 05m.tif": -0.5}
Hxx_acc = None
Hxy_acc = None
Hyy_acc = None
Ix_acc = None
Iy_acc = None
debug_images = {} # Stockage pour visu 3D
device = 'cuda' if torch.cuda.is_available() else 'cpu'
steger = StegerHessian(σ=σ_val, device=device)

for target_file, weight in weights.items():
    target_path = os.path.join(folder_path, target_file)
    if os.path.exists(target_path):
        print(f"Traitement de {target_file} (poids={weight})...")
        try:
            with rasterio.open(target_path) as src:
                data = src.read(1)
                # Normalisation Image
                data = data.astype(np.float32)
                if data.max() > 0:
                    data = (data - data.min()) / (data.max() - data.min())
                
                # Sauvegarde pour debug
                debug_images[target_file] = {'img': data.copy()}
                    
                # Conversion PyTorch
                img_tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(device)
                
                # Calcul Hessien brut
                ix, iy, ixx, ixy, iyy = steger.compute_hessian(img_tensor)
                λ1, λ2 = steger.compute_eigenvalues(ixx, ixy, iyy)
                
                debug_images[target_file]['l2'] = λ2.squeeze().cpu().numpy()
                
                # Affichage lambda2 individuel (avant normalisation)
                result_np = λ2.squeeze().cpu().numpy()
                plt.figure(figsize=(10, 8))
                plt.imshow(result_np, cmap='RdBu_r')
                plt.colorbar(label='Eigenvalue λ2')
                plt.title(f"Hessian Eigenvalue λ2 (σ={σ_val}) : {target_file}")
                plt.show()

                # Normalisation par quantile 99.9% (Robuste aux outliers)
                abs_lambda2 = torch.abs(λ2)
                max_val = torch.quantile(abs_lambda2, 0.999)
                if max_val > 0:
                    ix /= max_val
                    iy /= max_val
                    ixx /= max_val
                    ixy /= max_val
                    iyy /= max_val
                
                # Accumulation pondérée
                if Hxx_acc is None:
                    Ix_acc = weight * ix
                    Iy_acc = weight * iy
                    Hxx_acc = weight * ixx
                    Hxy_acc = weight * ixy
                    Hyy_acc = weight * iyy
                else:
                    Ix_acc += weight * ix
                    Iy_acc += weight * iy
                    Hxx_acc += weight * ixx
                    Hxy_acc += weight * ixy
                    Hyy_acc += weight * iyy
                    
        except Exception as e:
             print(f"Erreur sur {target_file}: {e}")
    else:
        print(f"Fichier cible {target_file} introuvable dans {folder_path}")

# Calcul des valeurs propres du Hessien fusionné
if Hxx_acc is not None:
    print("Calcul des valeurs propres du Hessien fusionné...")
    λ1_fused, λ2_fused = steger.compute_eigenvalues(Hxx_acc, Hxy_acc, Hyy_acc)
    
    result_np = λ2_fused.squeeze().cpu().numpy()
    
    plt.figure(figsize=(12, 10))
    plt.imshow(result_np, cmap='RdBu_r')
    plt.colorbar(label='Eigenvalue λ2 (Fused)')
    plt.title(f"Fused Hessian Eigenvalue λ2 (σ={σ_val})")
    plt.show()
    
    # --- 3. CONSTRUCTION DU GRAPHE SPARSE ET VISUALISATION ---
    # Force update graph_builder.py from GitHub to ensure latest version (debug stats)
    print("Mise à jour du script graph_builder.py...")
    !wget -q -O graph_builder.py https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/graph_builder.py

    # Import local
    import sys
    import os
    import importlib

    # Now that we download graph_builder.py in the first cell, simple import should work
    if os.path.exists('graph_builder.py'):
        import graph_builder as gb
    elif os.path.exists(os.path.join('Gauthier_Cerema', 'graph_builder.py')):
        sys.path.append(os.path.join(os.getcwd(), 'Gauthier_Cerema'))
        import graph_builder as gb
    else:
        # Fallback if file not found locally (maybe running without download cell?)
        try:
            import Gauthier_Cerema.graph_builder as gb
        except ImportError:
            print("Warning: graph_builder.py not found. Ensure the download cell was executed.")
            raise
            
    importlib.reload(gb)
    build_steger_graph = gb.build_steger_graph

    print("Construction du graphe sparse Steger (GPU)...")
    print(f"Plage λ2 fusionné : [{result_np.min():.4f}, {result_np.max():.4f}]")
    nodes, adj = build_steger_graph(Ix_acc, Iy_acc, Hxx_acc, Hxy_acc, Hyy_acc, R=10.0, τ=0.05, dark_ridges=True, steger_tolerance=1.0)
    
    if nodes is not None:
        # Filter: Keep connected nodes
        degrees = np.diff(adj.indptr)
        keep = np.where(degrees > 0)[0]
        
        if len(keep) > 0:
             min_dissims = [adj.data[adj.indptr[i]:adj.indptr[i+1]].min() for i in keep]
             node_response = np.array(min_dissims)
             coords = nodes['coords'][keep]
             directions = nodes['directions'][keep]
        else:
             print("Aucun noeud connecté.")
             node_response = np.array([])
             coords = np.zeros((0,2))
             directions = np.zeros((0,2))
        
        # Visualisation Interactive avec Plotly
        import plotly.graph_objects as go
        import plotly.figure_factory as ff
        
        # 1. Background Image (Heatmap of lambda2)
        # Flip vertically to match image coordinates usually (or rely on layout inversion)
        heatmap = go.Heatmap(z=result_np, colorscale='Gray', zmin=result_np.min(), zmax=result_np.max(), showscale=False, hoverinfo='skip')
        
        # 2. Nodes (Scatter points colored by Min Dissim)
        scatter_nodes = go.Scatter(
            x=coords[:, 0],
            y=coords[:, 1],
            mode='markers',
            marker=dict(
                size=3,
                color=node_response,
                colorscale='Viridis_r',
                colorbar=dict(title="Min Dissim"),
                opacity=0.8
            ),
            name='Steger Nodes',
            text=[f"Dissim: {v:.4f}" for v in node_response],
            hoverinfo='text'
        )

        # 3. Quiver (Arrows) - Subsampled if too many
        step = 1
        if len(coords) > 5000: step = 2
        if len(coords) > 15000: step = 5
        
        sub_x = coords[::step, 0]
        sub_y = coords[::step, 1]
        sub_u = directions[::step, 0]
        sub_v = directions[::step, 1]
        
        # Create Quiver figure (returns figure with data)
        quiver_fig = ff.create_quiver(sub_x, sub_y, sub_u, sub_v,
                                      scale=2.0, 
                                      arrow_scale=0.3,
                                      line=dict(width=1, color='cyan'))
        
        # Combine
        fig = go.Figure(data=[heatmap, scatter_nodes] + list(quiver_fig.data))
        
        fig.update_layout(
            title=f"Interactive Steger Graph (Nodes & Directions) - {len(coords)} nodes",
            width=1600, height=1600,
            yaxis=dict(scaleanchor="x", scaleratio=1, autorange='reversed'), # Image coords top-left origin
            plot_bgcolor='black'
        )
        fig.show()
    else:
        print("Aucun noeud extrait (vérifiez le seuil τ).")

    # --- NOUVEAU : VISUALISATION DES NOEUDS ACTIVÉS SEULS (Validation) ---
    if nodes is not None:
        print("Affichage des noeuds activés seuls (Validation Extraction)...")
        # On utilise les coordonnées subpixel brutes retournées
        px = nodes['coords'][:, 0]
        py = nodes['coords'][:, 1]
        
        plt.figure(figsize=(12, 12))
        # Fond : Lambda 2 fusionné
        plt.imshow(result_np, cmap='gray', vmin=result_np.min(), vmax=result_np.max())
        plt.colorbar(label='Lambda 2 (Fused)')
        
        # Points : Noeuds Steger
        plt.scatter(px, py, c='red', s=1, alpha=0.5, label='Steger Nodes')
        
        plt.title(f"Noeuds Steger Activés ({len(px)} points)")
        plt.legend()
        plt.axis('off')
        plt.show()

# --- 4. VISUALISATION 2D COMPARATIVE (Toute l'image) ---
if result_np is not None:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Subplots avec axes liés pour zoom synchro
    fig = make_subplots(rows=1, cols=3, 
                        shared_xaxes=True, shared_yaxes=True,
                        subplot_titles=("Ombrage (Brut)", "MNT (Brut)", "Hessien Fusionné (λ2)"),
                        horizontal_spacing=0.02)
    
    # 1. Ombrage
    if "Ombrage 05m base.tif" in debug_images:
        img = debug_images["Ombrage 05m base.tif"]['img']
        # Optimisation : Si image très grande, subsampling léger pour affichage fluide
        step = 1 if img.shape[0] < 2500 else 2
        fig.add_trace(go.Heatmap(z=img[::step, ::step], colorscale='Gray', showscale=False), row=1, col=1)
        
    # 2. MNT
    if "MNT 05m.tif" in debug_images:
        img = debug_images["MNT 05m.tif"]['img']
        step = 1 if img.shape[0] < 2500 else 2
        fig.add_trace(go.Heatmap(z=img[::step, ::step], colorscale='Viridis', showscale=False), row=1, col=2)

    # 3. Fused Hessien
    step = 1 if result_np.shape[0] < 2500 else 2
    fig.add_trace(go.Heatmap(z=result_np[::step, ::step], colorscale='RdBu_r', showscale=True), row=1, col=3)
    
    fig.update_layout(title="Comparaison des Modalités (Zoom Synchronisé)", height=800, width=1800)
    # Inverser Y pour correspondre aux coordonnées image
    fig.update_yaxes(autorange='reversed')
    fig.show()

# --- 5. ANALYSE INTERNE STEGER (GRADIENT & COURBURE PROJETÉS) ---
if Ix_acc is not None:
    print("Calcul des champs Steger (Gradient Normal & Courbure Normale)...")
    # Conversion CPU
    ix = Ix_acc.squeeze().cpu().numpy()
    iy = Iy_acc.squeeze().cpu().numpy()
    ixx = Hxx_acc.squeeze().cpu().numpy()
    ixy = Hxy_acc.squeeze().cpu().numpy()
    iyy = Hyy_acc.squeeze().cpu().numpy()
    
    # 1. Orientation Principale et Valeurs Propres (Hessien)
    # Formules exactes (Trace/Det) avec tri par valeur absolue
    trace = ixx + iyy
    disc = np.sqrt((ixx - iyy)**2 + 4 * ixy**2)
    
    l_plus = (trace + disc) / 2
    l_minus = (trace - disc) / 2
    
    # Angle de base
    theta = 0.5 * np.arctan2(2 * ixy, ixx - iyy)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    
    # Tri par valeur absolue pour trouver la normale (direction de courbure max)
    abs_l_plus = np.abs(l_plus)
    abs_l_minus = np.abs(l_minus)
    mask_minus_bigger = abs_l_minus > abs_l_plus
    
    # Si |l-| > |l+|, normale = (-sin, cos) associee a l-
    # Sinon, normale = (cos, sin) associee a l+
    nx = np.where(mask_minus_bigger, -sin_t, cos_t)
    ny = np.where(mask_minus_bigger, cos_t, sin_t)
    inn = np.where(mask_minus_bigger, l_minus, l_plus) # l2 (max curvature)
    
    # 2. Gradient projeté sur cette normale
    in_proj = ix * nx + iy * ny
    
    # Visualisation
    fig = make_subplots(rows=1, cols=3, 
                        shared_xaxes=True, shared_yaxes=True,
                        subplot_titles=("Gradient Normal (Doit traverser 0)", "Courbure Normale (Doit être Max)", "Hessien Original (λ2)"),
                        horizontal_spacing=0.02)
    
    # Subsampling si nécessaire
    step = 1 if ix.shape[0] < 2500 else 2
    
    # Gradient Normal (Divergent)
    # Echelle symétrique auto
    max_grad = np.max(np.abs(in_proj))
    fig.add_trace(go.Heatmap(z=in_proj[::step, ::step], colorscale='RdBu', zmin=-max_grad, zmax=max_grad, showscale=True), row=1, col=1)
    
    # Courbure Normale
    fig.add_trace(go.Heatmap(z=inn[::step, ::step], colorscale='Jet', showscale=True), row=1, col=2)
    
    # Rappel Lambda2
    fig.add_trace(go.Heatmap(z=result_np[::step, ::step], colorscale='RdBu_r', showscale=False), row=1, col=3)
    
    fig.update_layout(title="Analyse Interne Steger : Gradient vs Courbure", height=800, width=1800)
    fig.update_yaxes(autorange='reversed')
    fig.show()